In [0]:
# Silver layer -> convert types, standardize texts, remove duplicates and save data into the silver layers via MERGE.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "olist_lakehouse"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"

In [0]:
def to_ts(col_name: str):
    """Convert texts into timestamps; invalid values turns into null."""
    return F.expr(f"try_cast({col_name} AS TIMESTAMP)")

def to_int(col_name: str):
    return F.expr(f"try_cast({col_name} AS INT)")

def to_dec(col_name: str, precision: int = 10, scale: int = 2):
    return F.expr(f"try_cast({col_name} AS DECIMAL({precision},{scale}))")

def to_double(col_name: str):
    return F.expr(f"try_cast({col_name} AS DOUBLE)")

In [0]:
def normalize_city(col_name: str):
    """Lower case, no spaces, no blank spaces or accents."""
    return F.translate(
        F.lower(F.trim(F.col(col_name))),
        "áàâãäéèêëíìîïóòôõöúùûüç",
        "aaaaaeeeeiiiiooooouuuuc",
    )

def clean_text(col_name: str):
    """Remove blank spaces from edges and transforms empty spaces into null"""
    trimmed = F.trim(F.col(col_name))
    return F.when(trimmed == "", None).otherwise(trimmed)

In [0]:
def dedup_latest(df: DataFrame, keys: list[str], order_col: str = "_ingested_at") -> DataFrame:
    """Keeps only the latest record for each id."""
    window = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())
    return (
        df.withColumn("_rn", F.row_number().over(window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

In [0]:
def count_cast_failures(df: DataFrame, columns: list[str], cast_fn) -> dict:
    """Counts how many values there were originally mas turned into nulls after the conversion"""
    exprs = [
        F.sum((F.col(c).isNotNull() & cast_fn(c).isNull()).cast("int")).alias(c)
        for c in columns
    ]
    return df.select(exprs).first().asDict()

In [0]:
def upsert_silver(df: DataFrame, table: str, keys: list[str], comment: str) -> int:
    """Creates the table in the first execution; in the next ones runes the merges based on its key."""
    full_name = f"{SILVER}.{table}"
    df = df.withColumn("_processed_at", F.current_timestamp())

    if not spark.catalog.tableExists(full_name):
        df.write.format("delta").saveAsTable(full_name)
        spark.sql(f"COMMENT ON TABLE {full_name} IS '{comment}'")
    else:
        merge_condition = " AND ".join([f"t.{k} = s.{k}" for k in keys])
        (
            DeltaTable.forName(spark, full_name).alias("t")
            .merge(df.alias("s"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

    return spark.table(full_name).count()

In [0]:
bronze_orders = spark.table(f"{BRONZE}.orders")

date_cols = [
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]

print(count_cast_failures(bronze_orders, date_cols, to_ts))

In [0]:
bronze_orders.createOrReplaceTempView("orders_vw")

In [0]:
%sql
select * from orders_vw;

In [0]:
silver_orders = (
    bronze_orders
    .select(
        clean_text("order_id").alias("order_id"),
        clean_text("customer_id").alias("customer_id"),
        F.lower(clean_text("order_status")).alias("order_status"),
        to_ts("order_purchase_timestamp").alias("order_purchase_timestamp"),
        to_ts("order_approved_at").alias("order_approved_at"),
        to_ts("order_delivered_carrier_date").alias("order_delivered_carrier_date"),
        to_ts("order_delivered_customer_date").alias("order_delivered_customer_date"),
        to_ts("order_estimated_delivery_date").alias("order_estimated_delivery_date"),
        "_source_file",
        "_ingested_at",
    )
    .filter(F.col("order_id").isNotNull())
    .withColumn(
        "_dq_inconsistent_dates",
        F.coalesce(
            F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"),
            F.lit(False),
        ),
    )
)

silver_orders = dedup_latest(silver_orders, ["order_id"])

In [0]:
silver_order_items = (
    spark.table(f"{BRONZE}.order_items")
    .select(
        clean_text("order_id").alias("order_id"),
        to_int("order_item_id").alias("order_item_id"),
        clean_text("product_id").alias("product_id"),
        clean_text("seller_id").alias("seller_id"),
        to_ts("shipping_limit_date").alias("shipping_limit_date"),
        to_dec("price").alias("price"),
        to_dec("freight_value").alias("freight_value"),
        "_source_file",
        "_ingested_at",
    )
    .filter(F.col("order_id").isNotNull() & F.col("order_item_id").isNotNull())
)

silver_order_items = dedup_latest(silver_order_items, ["order_id", "order_item_id"])

In [0]:
silver_order_payments = (
    spark.table(f"{BRONZE}.order_payments")
    .select(
        clean_text("order_id").alias("order_id"),
        to_int("payment_sequential").alias("payment_sequential"),
        F.lower(clean_text("payment_type")).alias("payment_type"),
        to_int("payment_installments").alias("payment_installments"),
        to_dec("payment_value").alias("payment_value"),
        "_source_file",
        "_ingested_at",
    )
    .filter(F.col("order_id").isNotNull() & F.col("payment_sequential").isNotNull())
)

silver_order_payments = dedup_latest(silver_order_payments, ["order_id", "payment_sequential"])

In [0]:
silver_order_reviews = (
    spark.table(f"{BRONZE}.order_reviews")
    .select(
        clean_text("review_id").alias("review_id"),
        clean_text("order_id").alias("order_id"),
        to_int("review_score").alias("review_score"),
        clean_text("review_comment_title").alias("review_comment_title"),
        clean_text("review_comment_message").alias("review_comment_message"),
        to_ts("review_creation_date").alias("review_creation_date"),
        to_ts("review_answer_timestamp").alias("review_answer_timestamp"),
        "_source_file",
        "_ingested_at",
    )
    .filter(
        F.col("review_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("review_score").between(1, 5)
    )
)

silver_order_reviews = dedup_latest(silver_order_reviews, ["review_id", "order_id"])

In [0]:
silver_customers = dedup_latest(
    spark.table(f"{BRONZE}.customers")
    .select(
        clean_text("customer_id").alias("customer_id"),
        clean_text("customer_unique_id").alias("customer_unique_id"),
        F.lpad(clean_text("customer_zip_code_prefix"), 5, "0").alias("customer_zip_code_prefix"),
        normalize_city("customer_city").alias("customer_city"),
        F.upper(clean_text("customer_state")).alias("customer_state"),
        "_source_file",
        "_ingested_at",
    )
    .filter(F.col("customer_id").isNotNull()),
    ["customer_id"],
)

silver_sellers = dedup_latest(
    spark.table(f"{BRONZE}.sellers")
    .select(
        clean_text("seller_id").alias("seller_id"),
        F.lpad(clean_text("seller_zip_code_prefix"), 5, "0").alias("seller_zip_code_prefix"),
        normalize_city("seller_city").alias("seller_city"),
        F.upper(clean_text("seller_state")).alias("seller_state"),
        "_source_file",
        "_ingested_at",
    )
    .filter(F.col("seller_id").isNotNull()),
    ["seller_id"],
)

In [0]:
category_translation = (
    spark.table(f"{BRONZE}.category_translation")
    .select(
        clean_text("product_category_name").alias("product_category_name"),
        clean_text("product_category_name_english").alias("product_category_name_en"),
    )
)

silver_products = (
    spark.table(f"{BRONZE}.products")
    .select(
        clean_text("product_id").alias("product_id"),
        clean_text("product_category_name").alias("product_category_name"),
        to_int("product_name_lenght").alias("product_name_length"),
        to_int("product_description_lenght").alias("product_description_length"),
        to_int("product_photos_qty").alias("product_photos_qty"),
        to_int("product_weight_g").alias("product_weight_g"),
        to_int("product_length_cm").alias("product_length_cm"),
        to_int("product_height_cm").alias("product_height_cm"),
        to_int("product_width_cm").alias("product_width_cm"),
        "_source_file",
        "_ingested_at",
    )
    .filter(F.col("product_id").isNotNull())
    .join(category_translation, on="product_category_name", how="left")
    .withColumn("product_category_name", F.coalesce("product_category_name", F.lit("sem_categoria")))
    .withColumn("product_category_name_en", F.coalesce("product_category_name_en", F.lit("uncategorized")))
)

silver_products = dedup_latest(silver_products, ["product_id"])

In [0]:
silver_geolocation = (
    spark.table(f"{BRONZE}.geolocation")
    .select(
        F.lpad(clean_text("geolocation_zip_code_prefix"), 5, "0").alias("zip_code_prefix"),
        to_double("geolocation_lat").alias("lat"),
        to_double("geolocation_lng").alias("lng"),
        normalize_city("geolocation_city").alias("city"),
        F.upper(clean_text("geolocation_state")).alias("state"),
        "_ingested_at",
    )
    # does not consider coordinates outside of Brazil
    .filter(F.col("lat").between(-34, 6) & F.col("lng").between(-74, -34))
    .groupBy("zip_code_prefix")
    .agg(
        F.avg("lat").alias("lat"),
        F.avg("lng").alias("lng"),
        F.mode("city").alias("city"),    # most frequent city for the zipcode
        F.mode("state").alias("state"),
        F.max("_ingested_at").alias("_ingested_at"),
    )
)

In [0]:
# due to the lazy run of spark, everything will actually run in this cell

SILVER_TABLES = [
    (silver_orders,         "orders",         ["order_id"],                      "Pedidos tipados e deduplicados, com flag de datas inconsistentes"),
    (silver_order_items,    "order_items",    ["order_id", "order_item_id"],     "Itens dos pedidos com valores em DECIMAL"),
    (silver_order_payments, "order_payments", ["order_id", "payment_sequential"], "Pagamentos dos pedidos"),
    (silver_order_reviews,  "order_reviews",  ["review_id", "order_id"],         "Avaliações com nota validada entre 1 e 5"),
    (silver_customers,      "customers",      ["customer_id"],                   "Clientes com CEP e cidade padronizados"),
    (silver_sellers,        "sellers",        ["seller_id"],                     "Vendedores com CEP e cidade padronizados"),
    (silver_products,       "products",       ["product_id"],                    "Produtos com colunas corrigidas e categoria em inglês"),
    (silver_geolocation,    "geolocation",    ["zip_code_prefix"],               "Uma coordenada média por prefixo de CEP"),
]

for df, table, keys, comment in SILVER_TABLES:
    rows = upsert_silver(df, table, keys, comment)
    print(f"{SILVER}.{table:<16} {rows:>10,} linhas")

In [0]:
%sql
-- comparing counts between bronze and silver

SELECT 'orders' AS tabela,
       (SELECT COUNT(*) FROM olist_lakehouse.bronze.orders) AS bronze,
       (SELECT COUNT(*) FROM olist_lakehouse.silver.orders) AS silver
UNION ALL SELECT 'order_items',
       (SELECT COUNT(*) FROM olist_lakehouse.bronze.order_items),
       (SELECT COUNT(*) FROM olist_lakehouse.silver.order_items)
UNION ALL SELECT 'order_reviews',
       (SELECT COUNT(*) FROM olist_lakehouse.bronze.order_reviews),
       (SELECT COUNT(*) FROM olist_lakehouse.silver.order_reviews)
UNION ALL SELECT 'geolocation',
       (SELECT COUNT(*) FROM olist_lakehouse.bronze.geolocation),
       (SELECT COUNT(*) FROM olist_lakehouse.silver.geolocation)

In [0]:
%sql
-- checkin if there are no duplicate keys

SELECT order_id, COUNT(*) AS qtd
FROM olist_lakehouse.silver.orders
GROUP BY order_id
HAVING COUNT(*) > 1

In [0]:
%sql
-- checkin date data quality

SELECT _dq_inconsistent_dates, COUNT(*) AS qtd
FROM olist_lakehouse.silver.orders
GROUP BY _dq_inconsistent_dates

In [0]:
%sql
-- checking the merge record in the table history

DESCRIBE HISTORY olist_lakehouse.silver.orders